In [1]:
import pandas as pd
import sqlite3,os


In [ ]:

point=1316

def getClimateIndices(directory):
    temp = []  # temporary list/dict used throught the program 
 #   features = pd.DataFrame()  # this is a Pandas Dataframe that will hold the Climate Indices
    features={}
    #  add year-month pairs to the features dataframe that will match the available data
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file():
             #   print(entry.name)
    #  the Climate Indices are stored in flat files, so read them all in and store them in features
                file=entry.name
                file = file.rstrip("\n")
                spl = file.split(".")
                name = spl[0]
                temp = []
                tempdf = pd.DataFrame()
                print(file)
                if file.find("Zone.Identifier") == -1:
                    spl=file.split(".")
                    name=".".join(spl[:-1])
                    with open(f"{directory}/{file}","r") as fin:
                        line=fin.readline()
                        spl = line.split()
                        start = int(spl[0])
                        end = int(spl[1])
                        nl=0
                        print(start,end)
                        process=True
                        for nn in range(start,end+1):
                           line = fin.readline()
                           spl = line.split()
                           year = int(spl[0])
                           if year > 1949  and year < 2024:
                               for mon in range(1,13):
                                   val=float(spl[mon])
                                   if val < -30:
                                       process=False
                                       print(file,year,mon,val)
                        fin.close()
                    if process: 
                        with open(f"{directory}/{file}","r") as fin:
                            line=fin.readline()
                            spl = line.split()
                            start = int(spl[0])
                            end = int(spl[1])
                            nl=0
                            print(start,end)
                            process=True
                            for nn in range(start,end+1):
                               line = fin.readline()
                               spl = line.split()
                               year = int(spl[0])
                               if year not in features:
                                   features[year]={}
                                   
                               for mon in range(1,13):
                                   if mon not in features[year]:
                                       features[year][mon]={}
                                   val=float(spl[mon])
                                   features[year][mon][name]=val
                    else:
                        print("SKIPPING ",file)
                            
        return features

def getIndices(climate,start,end,indices=["all"]):
    features=[]
    features4Pbi=[]
    noUse={}
    indicesToUse={}

    frufru=[]
    for year,dct in climate.items():
        if year > 1989 and year < 2025:
            for mo,dct2 in dct.items():
                for ds,val in dct2.items():
                    if val < -30:
                        if ds not in noUse:
                            noUse[ds]=0
                        noUse[ds]+=1

    
    for year in range(start,end+1):
        dct=climate[year]

        for month,dct2 in dct.items():
            tmp=[]
            tmp2=[]
            tmp.append(year)
            tmp.append(month)
            tmp.append(f"{year}{str(month).zfill(2)}")
            for name,val in sorted(dct2.items()):
                if (indices[0]=="all" or name in indices) and name not in noUse:
                  tmp.append(val)
                  tmp2.append(val)
                
                  if name not in indicesToUse:
                      indicesToUse[name]=0
                  indicesToUse[name]+=1
            features4Pbi.append(tmp)
            features.append(tmp2)
    return features,indicesToUse,features4Pbi 

climate=getClimateIndices("Data/Climate-Indices")
features,indicesToUse,features4Pbi=getIndices(climate,1990,2024,["all"])


def readWxData(db):
    connr = sqlite3.connect(db)
    df=pd.read_sql('select * from MEANS_TTdRHVPD', connr)
    return df

def selectWxData(df,point,var="T"):    
    data=df.loc[df["Point"] == point]
    yrmos=df.loc[df["Point"] == point,"Yrmo"].astype(int).values
    mos=df.loc[df["Point"] == point,"Yrmo"].str[4:6].astype(int).values
    
    data=data[[var]].values.tolist()
    
    return data,yrmos,mos

df=readWxData('/home/joe/work/Fire/Data/DB/era5DataMeans.db')
df.drop(columns=['index'], inplace=True)
wxData,yrmos,mos=selectWxData(df,10,"VPD") 

dfM = df.loc[df["Point"] == point]
dfM["CDate"] = [int(dat[2]) for dat in features4Pbi]
for nn in range(1,7):
   dfM[f"F{nn}"] = [dat[nn+2] for dat in features4Pbi]

dfM.loc[dfM["Yrmo"].astype(int) != dfM["CDate"].astype(int)]
dfM.drop(columns=['CDate'], inplace=True)
outfile=f"full_data_point_{point}.csv"
dfM.to_csv(outfile, index=False)
print(f"Wrote {dfM.shape[0]} rows to {outfile}")


ea.data.txt
1948 2025
1948 2025
aao.data.txt
1979 2024
1979 2024
pna.data.txt
1948 2025
1948 2025
nino34.long.anom.data.txt
1870 2024
1870 2024
nao.long.data.txt
1821 2024
1821 2024
heatcentra.data.txt
1979 2025
1979 2025
nino12.long.anom.data.txt
1870 2024
1870 2024
Wrote 420 rows to full_data_point_1316.csv


/tmp/ipykernel_1971/2006633449.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfM["CDate"] = [int(dat[2]) for dat in features4Pbi]
/tmp/ipykernel_1971/2006633449.py:134: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfM[f"F{nn}"] = [dat[nn+2] for dat in features4Pbi]
/tmp/ipykernel_1971/2006633449.py:134: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pan

## Lat Lons 

In [2]:
def readWxData(db):
    connr = sqlite3.connect(db)
    df=pd.read_sql('select * from MEANS_TTdRHVPD', connr)
    return df

df=readWxData('/home/joe/work/Fire/Data/DB/era5DataMeans.db')
df.drop(columns=['index'], inplace=True)

In [3]:
df.head()

,Point,Yrmo,Length,T,Td,RH,VPD
0,0,199001,248,-8.463992,-12.366048,74.223548,0.092007
1,0,199002,224,-9.227723,-12.758839,76.263616,0.079737
2,0,199003,248,-3.466290,-7.204153,76.585927,0.127759
3,0,199004,240,4.388708,-3.163000,60.712417,0.380563
4,0,199005,248,7.292621,-4.213629,48.368589,0.628547


In [ ]:
latlons = pd.read_csv("latlons_42x71.csv")
latlons.rename(columns={"point":"Point"}, inplace=True)


In [5]:
latlons

,point,lat,lon
0,0,41.1,-109.0
1,1,41.1,-108.9
2,2,41.1,-108.8
3,3,41.1,-108.7
4,4,41.1,-108.6
...,...,...,...
2977,2977,37.0,-102.4
2978,2978,37.0,-102.3
2979,2979,37.0,-102.2
2980,2980,37.0,-102.1


In [14]:
pd.merge(df,latlons, on="Point").to_csv("era5_with_latlons.csv", index=False)

In [7]:
df.head()

,Point,Yrmo,Length,T,Td,RH,VPD
0,0,199001,248,-8.463992,-12.366048,74.223548,0.092007
1,0,199002,224,-9.227723,-12.758839,76.263616,0.079737
2,0,199003,248,-3.466290,-7.204153,76.585927,0.127759
3,0,199004,240,4.388708,-3.163000,60.712417,0.380563
4,0,199005,248,7.292621,-4.213629,48.368589,0.628547


In [8]:
latlons.head()

,point,lat,lon
0,0,41.1,-109.0
1,1,41.1,-108.9
2,2,41.1,-108.8
3,3,41.1,-108.7
4,4,41.1,-108.6


In [10]:
df.columns

Index(['Point', 'Yrmo', 'Length', 'T', 'Td', 'RH', 'VPD'], dtype='object')

In [11]:
latlons.columns

Index(['point', 'lat', 'lon'], dtype='object')